# Quant Factor Mining Agent developer example

In this notebook, we showcase how the NeMo Agent Toolkit can be used to create an end-to-end factor mining workflow for quantitative finance. This workflow demonstrates how to leverage LLMs to design an automated system to generate, code, and evaluate alpha factors.

## What You'll Learn

By the end of this notebook, you will understand how to:
- Set up a multi-step sequential workflow using NAT
- Implement custom functions for factor generation, code generation, and evaluation
- Use LLMs for creative factor design and code synthesis
- Evaluate factor performance using rank IC metrics
- Run complete end-to-end factor mining workflows with optimization loops

## Workflow Overview

This factor mining workflow consists of three agents orchestrated by the NeMo Agent Toolkit:

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🧩 Workflow Components</h4>
<ul>
<li><strong>Factor Agent</strong> — Uses an LLM to generate creative factor expressions based on price-volume data and predefined operators</li>
<li><strong>Code Agent</strong> — Converts factor expressions into executable Python code, leveraging the <code>Get_calculator</code> tool for operator implementations</li>
<li><strong>Eval Agent</strong> — Evaluates the predictive power of generated factors using Rank IC (backtesting) and provides optimization suggestions for iterative improvement</li>
</ul>
All three agents are powered by the <strong>nvidia/llama-3.3-nemotron-super-49b-v1.5</strong> NIM model.
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Note</h4>
This example demonstrates the capabilities of NeMo Agent Toolkit for automating quantitative research workflows. The generated factors should be thoroughly validated before use in production.
</div>

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing Dependencies](#installing-deps)
  - [0.4) Download Data](#download-data)
- [1) Understanding Factor Mining](#understanding-workflow)
  - [1.1) What is Factor Mining?](#what-is-factor-mining)
  - [1.2) Workflow Architecture](#workflow-architecture)
- [2) Source Code Components](#source-code)
  - [2.1) Factor Agent](#factor-agent)
  - [2.2) Code Agent](#code-agent)
  - [2.3) Eval Agent](#eval-agent) *(includes Rank IC utilities)*
  - [2.4) Factor Mining Optimization Workflow](#optimization-workflow)
  - [2.5) Register Module](#register-module)
- [3) Configuration](#configuration)
  - [3.1) Workflow Configuration](#workflow-config)
  - [3.2) Operator Templates](#operator-templates)
- [4) Running the Workflow](#running-workflow)
- [5) Interpreting Results](#interpreting-results)
  - [5.1) Understanding Evaluation Metrics](#evaluation-metrics)
  - [5.2) Analyzing Generated Factors](#analyzing-factors)
- [6) Conclusion](#conclusion)
- [7) Next Steps](#next-steps)

<h1 id="setup">0) Setup</h1>

<h2 id="prereqs">0.1) Prerequisites</h2>

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📋 Requirements</h4>
<ul>
<li><strong>Platform:</strong> Linux, macOS, or Windows</li>
<li><strong>Python:</strong> version 3.11, 3.12, or 3.13</li>
<li><strong>Package manager:</strong> pip or uv</li>
</ul>
</div>

<h2 id="api-keys">0.2) API Keys</h2>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔑 NVIDIA API Key Required</h4>
For this notebook, you will need an NVIDIA API key. Get yours from <a href="https://build.nvidia.com/settings/api-keys">build.nvidia.com</a>.
</div>

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<h2 id="installing-deps">0.3) Installing Dependencies</h2>

Install the factor mining workflow package:

In [ ]:
# Install the factor mining workflow package
%pip install -e ..

<h2 id="download-data">0.4) Download Data</h2>

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📥 S&P 500 Price-Volume Data</h4>
The workflow requires price-volume data (Open, Close, High, Low, Volume) stored as CSV files in <code>src/factor_mining_workflow/data/sp500/</code>. Use the script below to download fresh data from Yahoo Finance via <code>yfinance</code>.
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Disclaimer</h4>
Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
</div>

In [ ]:
from pathlib import Path
from factor_mining_workflow.download_data import download_sp500_data

data_dir = Path("../src/factor_mining_workflow/data/sp500")
if not data_dir.exists() or not list(data_dir.glob("*.csv")):
    print("Data not found. Downloading S&P 500 data...")
    data = download_sp500_data(start="2012-01-01", end="2025-12-31")
else:
    import pandas as pd
    print("Data already exists. Loading from disk...")
    data = {f.stem: pd.read_csv(f, index_col=0, parse_dates=True) for f in data_dir.glob("*.csv")}

for field, df in data.items():
    print(f"{field}: {df.shape[0]} days x {df.shape[1]} stocks")

<h1 id="understanding-workflow">1) Understanding Factor Mining</h1>

<h2 id="what-is-factor-mining">1.1) What is Factor Mining?</h2>

Factor mining is the process of discovering quantitative signals (factors) that have predictive power for future stock returns. In quantitative finance, factors are numerical characteristics of stocks that can be used to:

- **Predict returns**: Factors with high Information Coefficient (IC) can help forecast which stocks will outperform
- **Construct portfolios**: Long stocks with favorable factor values, short those with unfavorable values
- **Manage risk**: Understand exposures to different sources of returns

Traditional factor research is labor-intensive, requiring researchers to:
1. Formulate hypotheses about what might predict returns
2. Implement the factor calculation in code
3. Backtest the factor's performance
4. Iterate based on results

This workflow automates this process using LLMs!

<h2 id="workflow-architecture">1.2) Workflow Architecture</h2>

![Workflow Architecture](images/workflow-architecture.png)

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Closed-Loop Optimization</h4>
The workflow is orchestrated by the <strong>NeMo Agent Toolkit</strong> and uses a closed-loop optimization approach:
<ol>
<li><strong>Factor Agent</strong> receives price-volume data and operators, then generates factor expressions using the LLM</li>
<li><strong>Code Agent</strong> converts factor expressions into executable Python code, using the <code>Get_calculator</code> tool to access operator implementations</li>
<li><strong>Eval Agent</strong> evaluates the factor's predictive power via backtesting (Rank IC)
<ul>
<li>If IC meets threshold → Accept and produce <strong>backtesting results</strong></li>
<li>If IC is poor → Generate <strong>optimization suggestions</strong> and feed them back to the Factor Agent for a new iteration</li>
</ul>
</li>
</ol>
</div>

<h1 id="source-code">2) Source Code Components</h1>

Below we load and display the source code for each component of the factor mining workflow. Use `%load` to load the Python source files.

<h2 id="factor-agent">2.1) Factor Agent</h2>

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📊 Factor Agent</h4>
Uses the <strong>nvidia/llama-3.3-nemotron-super-49b-v1.5</strong> NIM to generate quantitative factor expressions based on available price-volume data (Open, Close, High, Low, Volume) and predefined operators.
<ul>
<li><strong>Input:</strong> Factor type request (e.g., "momentum factors"), price-volume data fields, operator definitions</li>
<li><strong>Output:</strong> Factor expressions in JSON format with names, formulas, and economic intuition</li>
</ul>
</div>

In [ ]:
# %load ../src/factor_mining_workflow/factor_generator.py

<h2 id="code-agent">2.2) Code Agent</h2>

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💻 Code Agent</h4>
Converts factor expressions into executable Python code. Uses the LLM to synthesize the factor function and leverages the <code>Get_calculator</code> tool to automatically include the required operator implementations from <code>calculator.json</code>.
<ul>
<li><strong>Input:</strong> Factor expressions (JSON) from the Factor Agent</li>
<li><strong>Output:</strong> Executable Python code implementing the factor calculations</li>
<li><strong>Tool:</strong> <code>Get_calculator</code> — retrieves operator implementations</li>
</ul>
</div>


In [ ]:
# %load ../src/factor_mining_workflow/factor_code_generator.py

<h2 id="eval-agent">2.3) Eval Agent</h2>

<div style="background-color: #fff3e0; border-left: 6px solid #ff9800; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📈 Eval Agent</h4>
Performs backtesting by computing the Spearman correlation (Rank IC) between factor values and forward stock returns to measure predictive power.
<ul>
<li><strong>Input:</strong> Executable codes from the Code Agent</li>
<li><strong>Output:</strong> Backtesting results (IC metrics) and optimization suggestions</li>
<li><strong>Accept criteria:</strong> |IC| ≥ threshold and p-value ≤ significance level</li>
<li><strong>Feedback loop:</strong> When factors fall below threshold, generates optimization suggestions for the Factor Agent</li>
</ul>
</div>

In [ ]:
# %load ../src/factor_mining_workflow/factor_evaluator.py

<h2 id="optimization-workflow">2.4) Factor Mining Optimization Workflow</h2>

<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="margin-top: 0;">🔄 NeMo Agent Toolkit Orchestration</h4>
The Factor Mining Optimization Workflow uses the NeMo Agent Toolkit to orchestrate the entire closed-loop pipeline. It composes the Factor Agent, Code Agent, and Eval Agent — iteratively generating factor expressions, converting them to executable codes, backtesting their performance, and feeding optimization suggestions back until the IC threshold is met.
<pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; margin: 10px 0;">
Factor Agent → factor expressions → Code Agent → executable codes → Eval Agent
      ↑                                                                    |
      └──────────── optimization suggestions ←─────────────────────────────┘
</pre>
</div>

In [ ]:
# %load ../src/factor_mining_workflow/factor_mining_optimization_workflow.py

<h2 id="register-module">2.5) Register Module</h2>

The register module imports all components to make them available to the NeMo Agent Toolkit.

In [ ]:
# %load ../src/factor_mining_workflow/register.py

<h1 id="configuration">3) Configuration</h1>

<h2 id="workflow-config">3.1) Workflow Configuration</h2>

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚙️ Workflow Configuration</h4>
The workflow configuration defines how the factor optimization agent operates. It specifies:
<ul>
<li><strong>LLM assignments</strong> — Which model powers each agent (Factor, Code, Eval)</li>
<li><strong>Temperature settings</strong> — Higher for creative factor generation, lower for precise code generation</li>
<li><strong>Acceptance criteria</strong> — IC threshold and p-value thresholds for factor quality</li>
<li><strong>Iteration limits</strong> — Maximum optimization attempts before accepting best result</li>
</ul>
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ LLM Base URL</h4>
The <code>base_url</code> for the LLMs depends on your API key:
<ul>
<li><code>https://integrate.api.nvidia.com/v1/</code> — for keys from <a href="https://build.nvidia.com">build.nvidia.com</a></li>
<li><code>https://inference-api.nvidia.com/v1/</code> — for NVIDIA internal or enterprise API keys</li>
</ul>
Update the <code>base_url</code> field in <code>configs/config-optimization.yml</code> to match your API key type.
</div>

In [ ]:
# Display the configuration file
with open("../configs/config-optimization.yml", "r") as f:
    print(f.read())

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Configuration Parameters Explained</h4>
<table style="width: 100%; border-collapse: collapse;">
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>factor_generator_llm</code></td><td style="padding: 8px;">LLM for generating factor expressions (higher temperature for creativity)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>code_generator_llm</code></td><td style="padding: 8px;">LLM for converting expressions to executable code (lower temperature for precision)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>optimization_advisor_llm</code></td><td style="padding: 8px;">LLM for generating optimization feedback (balanced temperature)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>ic_threshold</code></td><td style="padding: 8px;">Minimum absolute IC value to accept a factor (e.g., 0.02 = 2%)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>p_value_threshold</code></td><td style="padding: 8px;">Maximum p-value for statistical significance (e.g., 0.05 = 5%)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>max_iterations</code></td><td style="padding: 8px;">Maximum number of optimization iterations before accepting best result</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>num_factors</code></td><td style="padding: 8px;">Number of factors to generate per iteration</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>forward_periods</code></td><td style="padding: 8px;">Number of days for forward return calculation (e.g., 5 = weekly)</td></tr>
<tr><td style="padding: 8px;"><code>save_results</code></td><td style="padding: 8px;">Whether to save successful factors to disk</td></tr>
</table>
</div>

<h2 id="operator-templates">3.2) Operator Templates</h2>

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🧮 Operator Templates</h4>
The workflow uses predefined operators from <code>calculator.json</code>. These operators are the building blocks that the Factor Agent uses to compose factor expressions. The <code>Get_calculator</code> tool provides these implementations to the Code Agent.
</div>

In [ ]:
import json
from pathlib import Path

# Load and display sample operators
template_path = Path("../src/factor_mining_workflow/template/calculator.json")
with open(template_path, "r") as f:
    operators = json.load(f)

print(f"Total operators available: {len(operators)}\n")
print("Sample operators:")
print("-" * 60)
for op in operators[:10]:
    print(f"\n{op['name']}:")
    print(f"  Signature: {op['signature']}")
    print(f"  Meaning: {op['meanings'][:80]}...")

<h1 id="running-workflow">4) Running the Workflow</h1>

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 Running the Workflow</h4>
Now we run the workflow and see the generated factors. The workflow will:
<ol>
<li>Start the NeMo Agent Toolkit orchestration</li>
<li>Load price-volume data (Open, Close, High, Low, Volume)</li>
<li>Run the Factor Agent → Code Agent → Eval Agent pipeline</li>
<li>Iterate until IC threshold is met or max iterations reached</li>
<li>Save accepted factors to disk</li>
</ol>
</div>

Phoenix is an open-source observability platform that captures and visualizes traces from your agent. It runs as a local web server that collects trace data and provides an interactive UI for analysis.
<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 Production vs. Notebook Setup</h4>
<strong>In production,</strong> you'd simply run in a terminal:
<pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; margin: 10px 0;">
phoenix serve
</pre>
<strong>In Jupyter,</strong> you'll use <code>subprocess</code> to start Phoenix in the background.
</div>

In [ ]:
import subprocess
import time
import sys

# Start Phoenix 
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Read initial output
print("Starting Phoenix...")
start_time = time.time()
while time.time() - start_time < 3:
    line = phoenix_process.stdout.readline()
    if line:
        print(line.strip())

print("\n✅ Phoenix is running silently in the background!")

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 What Phoenix Does</h4>
<ul>
<li><strong>Captures traces</strong> - Every agent decision, tool call, and LLM interaction</li>
<li><strong>Stores telemetry</strong> - Timing data, token usage, success/failure status</li>
<li><strong>Provides visualization</strong> - Interactive UI to explore traces and find patterns</li>
<li><strong>Runs locally</strong> - Your data never leaves your machine (default port: 6006)</li>
</ul>
</div>

## Access Phoenix UI

If the button below does not work, manually open <a href="http://localhost:6006">http://localhost:6006</a> in your browser.

In [ ]:
from IPython.display import HTML, display

html = """
<div style="padding: 15px; background-color: #e8f4f8; border-radius: 8px; margin: 10px 0;">
    <h4 style="margin: 0 0 10px 0;">🔍 Phoenix Observability Dashboard</h4>
    <p>Click the button to open Phoenix (it will auto-detect the correct URL):</p>
    <button onclick="
        var url = window.location.origin.replace(/p\\d+/, 'p6006');
        window.open(url, '_blank');
    " style="padding: 10px 20px; background-color: #0066cc; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 14px;">
        🚀 Open Phoenix UI
    </button>
    <p style="margin-top: 10px; font-size: 12px; color: #666;">
        If the button doesn't work, open <a href="http://localhost:6006">http://localhost:6006</a> directly
    </p>
</div>
"""

display(HTML(html))

## Phoenix Configuration
To enable Phoenix tracing, you add a `telemetry section` to your NAT config. This tells NAT where to send trace data.
<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="text-align: center;">Phoenix Configuration Structure</h4>
<pre style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin: 10px 0;">
general:                                              # Global workflow settings
  telemetry:                                          # Monitoring and metrics
    tracing:                                          # Distributed tracing config
      phoenix:                                        # Phoenix-specific settings
        _type: phoenix                                # Use Phoenix provider
        endpoint: http://localhost:6006/v1/traces     # Where to send data
        project: factor_mining_workflow               # Project name in UI
</pre>
</div>
<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Key Configuration Fields</h4>
<ul>
<li><strong>endpoint</strong> - Phoenix trace collector URL (must match where Phoenix is running)</li>
<li><strong>project</strong> - Groups traces together in the UI. Use different names to compare versions (e.g., "baseline" vs. "optimized")</li>
<li><strong>_type: phoenix</strong> - NAT supports multiple tracing backends; this specifies Phoenix</li>
</ul>
</div>
The config file at `configs/config-optimization.yml` already includes this telemetry section.

In [ ]:
# Generate momentum factors
!cd .. && nat run --config_file configs/config-optimization.yml --input "momentum factors"

### Running with Different Factor Types

The same workflow handles any factor category — just change the `--input` string. The Factor Agent will generate completely different formulas appropriate for each category.

```bash
# Volatility factors (e.g., rolling standard deviation, range-based)
nat run --config_file configs/config-optimization.yml --input "volatility factors"

# Mean-reversion factors (e.g., distance from moving average)
nat run --config_file configs/config-optimization.yml --input "mean-reversion factors"

# Volume-price divergence factors (e.g., volume relative to price moves)
nat run --config_file configs/config-optimization.yml --input "volume price divergence factors"
```

You can run any of these from a terminal in the project root, or invoke them from this notebook with `!nat run ...` if you want to keep iterating interactively.

<h1 id="interpreting-results">5) Interpreting Results</h1>

<h2 id="evaluation-metrics">5.1) Understanding Evaluation Metrics</h2>

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📊 Backtesting Metrics</h4>
The Eval Agent evaluates factors using several key metrics:
<table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
<tr style="background-color: #bbdefb;"><th style="padding: 8px; text-align: left;">Metric</th><th style="padding: 8px; text-align: left;">Description</th><th style="padding: 8px; text-align: left;">Good Value</th></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>Mean IC</strong></td><td style="padding: 8px;">Average Spearman correlation between factor and forward returns</td><td style="padding: 8px;">|IC| > 0.03</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>IC Std</strong></td><td style="padding: 8px;">Standard deviation of IC values</td><td style="padding: 8px;">Lower is more consistent</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>IC IR</strong></td><td style="padding: 8px;">Information Ratio = Mean IC / IC Std</td><td style="padding: 8px;">> 0.5 is good</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>T-statistic</strong></td><td style="padding: 8px;">Statistical significance of mean IC</td><td style="padding: 8px;">|t| > 2 is significant</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>P-value</strong></td><td style="padding: 8px;">Probability IC is different from zero</td><td style="padding: 8px;">< 0.05 is significant</td></tr>
<tr><td style="padding: 8px;"><strong>Positive IC Ratio</strong></td><td style="padding: 8px;">Fraction of periods with positive IC</td><td style="padding: 8px;">> 0.55 is good</td></tr>
</table>
</div>

<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="margin-top: 0;">📈 Rank IC Value Interpretation</h4>
<table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
<tr style="background-color: #ffcdd2;"><td style="padding: 10px; width: 20%;"><strong>< 0.02</strong></td><td style="padding: 10px;"><strong>Weak/Noise:</strong> Very little predictive power. May still be useful in a multi-factor model, but not a strong standalone signal.</td></tr>
<tr style="background-color: #c8e6c9;"><td style="padding: 10px;"><strong>0.02 – 0.05</strong></td><td style="padding: 10px;"><strong>Good/Standard:</strong> Target range for institutional-grade factors (Value, Quality). Generates consistent Alpha across a large stock universe.</td></tr>
<tr style="background-color: #a5d6a7;"><td style="padding: 10px;"><strong>0.05 – 0.15</strong></td><td style="padding: 10px;"><strong>Strong:</strong> Very high-quality signal, typical of short-term Momentum or specialized High-Frequency factors.</td></tr>
<tr style="background-color: #fff3cd;"><td style="padding: 10px;"><strong>> 0.15</strong></td><td style="padding: 10px;"><strong>Exceptional/Suspicious:</strong> Rare in live markets. Check for look-ahead bias or overfitting.</td></tr>
</table>
</div>

<h2 id="analyzing-factors">5.2) Analyzing Generated Factors</h2>

The cell below automatically loads the most recent factor output and displays the generated factor expressions, which factor was selected for evaluation, and the corresponding IC metrics.

In [ ]:
import json
import re
from pathlib import Path

def _match_factor_name(func_name: str, display_name: str) -> bool:
    """Check if a Python function name corresponds to a factor's display name."""
    func_tokens = set(func_name.lower().replace("factor_", "").split("_"))
    display_tokens = set(re.split(r"[\s\-_]+", display_name.lower()))
    return func_tokens.issubset(display_tokens)

def _parse_factor_expressions(factor_description: str) -> list:
    """Extract the factor JSON array from the LLM's factor_description output."""
    match = re.search(r"```json\s*(\[.*?\])\s*```", factor_description, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(factor_description)
    except (json.JSONDecodeError, TypeError):
        return []

def interpret_ic_results(result_path: str):
    """
    Interpret the IC evaluation results from a saved factor JSON file.
    """
    with open(result_path, "r", encoding="utf-8") as f:
        result = json.load(f)
    
    print("=" * 60)
    print("FACTOR EVALUATION RESULTS")
    print("=" * 60)
    
    print(f"\nTimestamp: {result.get('timestamp', 'N/A')}")
    print(f"Iteration: {result.get('iteration', 'N/A')}")
    
    selected = result.get("selected_factor", "")
    
    factors = _parse_factor_expressions(result.get("factor_description", ""))
    if factors:
        print(f"\nGenerated Factors ({len(factors)}):")
        print("-" * 40)
        for i, factor in enumerate(factors, 1):
            name = factor.get('name', 'N/A')
            is_selected = selected and _match_factor_name(selected, name)
            marker = " << SELECTED" if is_selected else ""
            print(f"\n  Factor {i}: {name}{marker}")
            print(f"    Formula:    {factor.get('formula', 'N/A')}")
            print(f"    Meaning:    {factor.get('meaning', 'N/A')}")
            print(f"    Category:   {factor.get('category', 'N/A')}")
            print(f"    Data fields: {', '.join(factor.get('data_fields_used', []))}")
            print(f"    Operators:   {', '.join(factor.get('operators_used', []))}")
            print(f"    Lookback:    {factor.get('lookback_periods', 'N/A')}")
    
    metrics = result.get("evaluation_metrics", {})
    if metrics:
        metric_label = f"Evaluation Metrics (on: {selected})" if selected else "Evaluation Metrics"
        print(f"\n{metric_label}:")
        print("-" * 40)
        print(f"  Mean IC: {metrics.get('mean_ic', 'N/A'):.4f}" if metrics.get('mean_ic') is not None else "  Mean IC: N/A")
        print(f"  IC Std: {metrics.get('ic_std', 'N/A'):.4f}" if metrics.get('ic_std') is not None else "  IC Std: N/A")
        print(f"  IC IR: {metrics.get('ic_ir', 'N/A'):.4f}" if metrics.get('ic_ir') is not None else "  IC IR: N/A")
        print(f"  T-stat: {metrics.get('t_stat', 'N/A'):.4f}" if metrics.get('t_stat') is not None else "  T-stat: N/A")
        print(f"  P-value: {metrics.get('p_value', 'N/A'):.6f}" if metrics.get('p_value') is not None else "  P-value: N/A")
        print(f"  Num Periods: {metrics.get('num_periods', 'N/A')}")
        print(f"  Positive IC Ratio: {metrics.get('positive_ic_ratio', 'N/A'):.2%}" if metrics.get('positive_ic_ratio') is not None else "  Positive IC Ratio: N/A")
        
        percentiles = metrics.get("ic_percentiles", {})
        if percentiles:
            print("\n  IC Percentiles:")
            for pct, val in percentiles.items():
                print(f"    {pct}: {val:.4f}")
    
    factor_code = result.get("factor_code", "")
    if factor_code:
        print("\nFactor Code:")
        print("-" * 40)
        print(factor_code)

# Automatically pick the most recent output file
output_dir = Path("../src/factor_mining_workflow/output")
factor_files = sorted(output_dir.glob("factor_*.json"), key=lambda f: f.stat().st_mtime, reverse=True)
if factor_files:
    print(f"Showing results for: {factor_files[0].name}\n")
    interpret_ic_results(str(factor_files[0]))
else:
    print("No factor output files found. Run the workflow first!")

<h1 id="conclusion">6) Conclusion</h1>

<div style="background-color: #fff8e1; border-left: 6px solid #ffa000; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ About This Example Run</h4>
<p>
The results shown in this notebook are from a <strong>demonstration run</strong> with <code>max_iterations: 2</code>. This means the optimization loop stops after at most <strong>two iterations</strong> — if the generated factor does not meet the IC threshold on the first attempt, the workflow collects optimization feedback and tries once more before accepting the best result found so far (a "best effort" outcome).
</p>
<p>
In practice, you can increase <code>max_iterations</code> in the workflow configuration to allow the agent more attempts to discover higher-quality factors. More iterations give the LLM additional chances to refine its factor formulas based on evaluation feedback, which often leads to stronger signals. However, each iteration incurs additional LLM inference cost and runtime, so the right setting depends on your time and compute budget.
</p>
</div>

<h1 id="next-steps">7) Next Steps</h1>

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔮 Ideas for Extending the Workflow</h4>
<ol>
<li><strong>Add more operators</strong> — Extend <code>calculator.json</code> with additional technical indicators</li>
<li><strong>Use different data</strong> — Replace SP500 data with your own stock universe</li>
<li><strong>Customize evaluation</strong> — Modify IC thresholds or add additional metrics like turnover</li>
<li><strong>Add more data fields</strong> — Include fundamental data like earnings, book value, etc.</li>
<li><strong>Build factor portfolios</strong> — Use accepted factors to construct trading strategies</li>
</ol>
</div>

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📚 Additional Resources</h4>
<ul>
<li><a href="https://docs.nvidia.com/nemo-agent-toolkit/">NeMo Agent Toolkit Documentation</a></li>
<li><a href="https://arize.com/docs/phoenix">Arize Phoenix Documentation</a></li>
</ul>
</div>